# 01 — UniTox Dataset Audit & Baseline Setup

This notebook is the first step of the Reliability-Aware UniTox project.

## Goal

Before changing the UniTox method, we will:

1. Load the official small-molecule benchmark.
2. Audit its structure and labels.
3. Verify the supplied Morgan/RDKit fingerprint files.
4. Reproduce the UniTox Random Forest configuration for one task:
   - Liver toxicity
   - Binary label
   - Morgan fingerprint
   - 10-fold cross-validation
   - ROC-AUC

**Reference configuration from the UniTox repository:** Morgan fingerprint = 2048 bits, radius = 2, Random Forest = 500 trees, seed = 0, 10-fold CV, metric = AUC.


In [1]:
import os

# Folder containing the UniTox repository
PROJECT_ROOT = r"D:\pro dataset\UniTox-main"

DATA_DIR = os.path.join(PROJECT_ROOT, "Data")

CSV_PATH = os.path.join(
    DATA_DIR,
    "UniTox-Small-Molecule-Benchmark.csv"
)

MORGAN_PATH = os.path.join(
    DATA_DIR,
    "UniTox-Small-Molecule-Benchmark-morgan.npz"
)

RDKIT_PATH = os.path.join(
    DATA_DIR,
    "UniTox-Small-Molecule-Benchmark-rdkit.npz"
)

In [2]:
import os
import numpy as np
import pandas as pd

print("CSV exists:", os.path.exists(CSV_PATH))
print("Morgan NPZ exists:", os.path.exists(MORGAN_PATH))
print("RDKit NPZ exists:", os.path.exists(RDKIT_PATH))


CSV exists: True
Morgan NPZ exists: True
RDKit NPZ exists: True


In [3]:
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nColumns:")
for i, c in enumerate(df.columns, 1):
    print(f"{i:2d}. {c}")


Shape: (1349, 43)

Columns:
 1. generic_name
 2. cardio_toxicity_reasoning
 3. cardio_toxicity_ternary_rating
 4. cardio_toxicity_binary_rating
 5. cardio_toxicity_confident_ternary_rating_0_1
 6. cardio_toxicity_binary_rating_0_1
 7. dermatological_toxicity_reasoning
 8. dermatological_toxicity_ternary_rating
 9. dermatological_toxicity_binary_rating
10. dermatological_toxicity_confident_ternary_rating_0_1
11. dermatological_toxicity_binary_rating_0_1
12. hematological_reasoning
13. hematological_ternary_rating
14. hematological_binary_rating
15. hematological_confident_ternary_rating_0_1
16. hematological_binary_rating_0_1
17. infertility_reasoning
18. infertility_ternary_rating
19. infertility_binary_rating
20. infertility_confident_ternary_rating_0_1
21. infertility_binary_rating_0_1
22. liver_toxicity_reasoning
23. liver_toxicity_ternary_rating
24. liver_toxicity_binary_rating
25. liver_toxicity_confident_ternary_rating_0_1
26. liver_toxicity_binary_rating_0_1
27. ototoxicity_reas

In [4]:
print("Unique drugs:", df["generic_name"].nunique())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate drug names:", df["generic_name"].duplicated().sum())
print("Missing SMILES:", df["smiles"].isna().sum())
print("Missing all_smiles:", df["all_smiles"].isna().sum())
print("Rows:", len(df))


Unique drugs: 1349
Duplicate rows: 0
Duplicate drug names: 0
Missing SMILES: 0
Missing all_smiles: 0
Rows: 1349


In [5]:
toxicity_targets = [
    "cardio_toxicity",
    "dermatological_toxicity",
    "hematological",
    "infertility",
    "liver_toxicity",
    "ototoxicity",
    "pulmonary_toxicity",
    "renal_toxicity",
]

summary = []
for tox in toxicity_targets:
    ternary = f"{tox}_ternary_rating"
    binary = f"{tox}_binary_rating"
    confident = f"{tox}_confident_ternary_rating_0_1"
    binary01 = f"{tox}_binary_rating_0_1"

    summary.append({
        "toxicity": tox,
        "ternary_missing": df[ternary].isna().sum(),
        "binary_missing": df[binary].isna().sum(),
        "confident_missing": df[confident].isna().sum(),
        "binary01_missing": df[binary01].isna().sum(),
        "binary_yes": (df[binary] == "Yes").sum(),
        "binary_no": (df[binary] == "No").sum(),
        "ternary_less": (df[ternary] == "Less").sum(),
        "ternary_most": (df[ternary] == "Most").sum(),
        "ternary_no": (df[ternary] == "No").sum(),
    })

pd.DataFrame(summary)


,toxicity,ternary_missing,binary_missing,confident_missing,binary01_missing,binary_yes,binary_no,ternary_less,ternary_most,ternary_no
0,cardio_toxicity,0,0,448,0,755,594,429,594,307
1,dermatological_toxicity,0,0,394,0,1199,150,394,828,127
2,hematological,0,0,274,0,876,473,274,655,420
3,infertility,0,0,328,0,503,846,328,336,685
4,liver_toxicity,0,0,471,0,736,613,471,481,397
5,ototoxicity,0,0,166,0,121,1228,166,54,1129
6,pulmonary_toxicity,0,0,352,0,498,851,352,357,640
7,renal_toxicity,0,0,401,0,508,841,401,369,579


In [6]:
tox = "liver_toxicity"

print("Liver ternary:")
print(df[f"{tox}_ternary_rating"].value_counts(dropna=False))

print("\nLiver binary:")
print(df[f"{tox}_binary_rating"].value_counts(dropna=False))

print("\nLiver confident 0/1:")
print(df[f"{tox}_confident_ternary_rating_0_1"].value_counts(dropna=False))

print("\nLiver binary 0/1:")
print(df[f"{tox}_binary_rating_0_1"].value_counts(dropna=False))


Liver ternary:
liver_toxicity_ternary_rating
Most    481
Less    471
No      397
Name: count, dtype: int64

Liver binary:
liver_toxicity_binary_rating
Yes    736
No     613
Name: count, dtype: int64

Liver confident 0/1:
liver_toxicity_confident_ternary_rating_0_1
1.0    481
NaN    471
0.0    397
Name: count, dtype: int64

Liver binary 0/1:
liver_toxicity_binary_rating_0_1
1    736
0    613
Name: count, dtype: int64


In [7]:
morgan = np.load(MORGAN_PATH)["features"]
rdkit = np.load(RDKIT_PATH)["features"]

print("Morgan shape:", morgan.shape, "dtype:", morgan.dtype)
print("RDKit shape:", rdkit.shape, "dtype:", rdkit.dtype)


Morgan shape: (1349, 2048) dtype: float32
RDKit shape: (1349, 200) dtype: float32


In [8]:
pip install rdkit

Note: you may need to restart the kernel to use updated packages.


## Important observation

The supplied fingerprint files have the same row count as the benchmark:

- Morgan: **1349 × 2048**
- RDKit: **1349 × 200**

Therefore, row order must be preserved when joining fingerprints to the CSV.

We will verify this against independently generated RDKit fingerprints before using the supplied arrays as our baseline features.


In [9]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import numpy as np

# Create Morgan fingerprint generator
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

def morgan_fp_array(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    fp = morgan_generator.GetFingerprint(mol)

    arr = np.zeros((2048,), dtype=np.int8)

    # Convert RDKit fingerprint to NumPy array
    from rdkit import DataStructs
    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr

In [10]:
# Compare all rows with the supplied Morgan fingerprint.
# This can take a little time.

generated = np.vstack([
    morgan_fp_array(s)
    for s in df["smiles"]
])

print("Generated shape:", generated.shape)
print("Supplied shape:", morgan.shape)

same = np.array_equal(generated, morgan)
print("Exact array equality:", same)

if not same:
    differing_rows = np.where(np.any(generated != morgan, axis=1))[0]
    print("Rows differing:", len(differing_rows))
    print("First differing row:", differing_rows[:10])


[16:52:00] WARNING: not removing hydrogen atom without neighbors
[16:52:00] WARNING: not removing hydrogen atom without neighbors
[16:52:00] WARNING: not removing hydrogen atom without neighbors
[16:52:00] WARNING: not removing hydrogen atom without neighbors
[16:52:00] WARNING: not removing hydrogen atom without neighbors
[16:52:00] WARNING: not removing hydrogen atom without neighbors
[16:52:00] WARNING: not removing hydrogen atom without neighbors
[16:52:00] WARNING: not removing hydrogen atom without neighbors
[16:52:00] WARNING: not removing hydrogen atom without neighbors
[16:52:00] WARNING: not removing hydrogen atom without neighbors


Generated shape: (1349, 2048)
Supplied shape: (1349, 2048)
Exact array equality: True


[16:52:01] WARNING: not removing hydrogen atom without neighbors
[16:52:01] WARNING: not removing hydrogen atom without neighbors


# First baseline: Liver + Binary + Morgan + Random Forest

This follows the UniTox reference configuration:

- Target: `liver_toxicity_binary_rating_0_1`
- Features: Morgan fingerprint
- Bits: 2048
- Radius: 2
- Random Forest: 500 trees
- Seed: 0
- CV: 10 folds
- Metric: ROC-AUC

The published repository result we will use as our reference is approximately:

**ROC-AUC = 0.7631 ± 0.0321**

Our reproduction does not need to match every fold bit-for-bit unless the original split implementation is identical; it should be reasonably close before we proceed.


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

target = "liver_toxicity_binary_rating_0_1"

X = morgan
y = df[target].astype(int).to_numpy()

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=0,
    n_jobs=-1
)

scores = cross_val_score(
    rf,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("Fold AUCs:")
print(scores)
print("\nMean AUC:", scores.mean())
print("Std AUC:", scores.std())


Fold AUCs:
[0.69200177 0.76347768 0.73840035 0.82310589 0.78511298 0.78001772
 0.76517501 0.83340718 0.79209127 0.72557826]

Mean AUC: 0.7698368125012359
Std AUC: 0.040961683135931265


# Reference comparison

UniTox repository reference for:

`Random Forest + Morgan + Binary + 10-fold CV + Liver`

**0.7630554777 ± 0.0320785795**

If our result differs noticeably, do **not** modify the model to force agreement. Instead, we will inspect the original split implementation and data preparation.

That reproducibility check comes before our proposed reliability-aware model.


In [12]:
same = np.array_equal(generated, morgan)

print("Exact array equality:", same)

if not same:
    differing_rows = np.where(
        np.any(generated != morgan, axis=1)
    )[0]

    print("Rows differing:", len(differing_rows))
    print("First differing rows:", differing_rows[:20])

Exact array equality: True


In [13]:
confident_target = "liver_toxicity_confident_ternary_rating_0_1"

mask = df[confident_target].notna()

X_conf = morgan[mask]
y_conf = df.loc[mask, confident_target].astype(int).to_numpy()

print("Total samples:", len(df))
print("Confident samples:", len(y_conf))
print("Removed samples:", len(df) - len(y_conf))

print("\nClass distribution:")
print(pd.Series(y_conf).value_counts())

Total samples: 1349
Confident samples: 878
Removed samples: 471

Class distribution:
1    481
0    397
Name: count, dtype: int64


In [14]:
rf_conf = RandomForestClassifier(
    n_estimators=500,
    random_state=0,
    n_jobs=-1
)

cv_conf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

scores_conf = cross_val_score(
    rf_conf,
    X_conf,
    y_conf,
    cv=cv_conf,
    scoring="roc_auc",
    n_jobs=-1
)

print("Confident-label fold AUCs:")
print(scores_conf)

print("\nMean AUC:", scores_conf.mean())
print("Std AUC:", scores_conf.std())

Confident-label fold AUCs:
[0.82369792 0.8015625  0.8296875  0.82473958 0.76692708 0.8171875
 0.85130208 0.83150183 0.84161325 0.91773504]

Mean AUC: 0.8305954288766788
Std AUC: 0.03647372262303214


In [15]:
same = np.array_equal(generated, morgan)

print("Exact array equality:", same)

if not same:
    differing_rows = np.where(
        np.any(generated != morgan, axis=1)
    )[0]

    print("Rows differing:", len(differing_rows))
    print("First differing rows:", differing_rows[:20])

Exact array equality: True


# Next experiments

After this notebook passes:

1. Reproduce all 8 toxicity tasks.
2. Reproduce the confident-label version.
3. Compare binary vs confident-label performance.
4. Add scaffold-balanced evaluation.
5. Inspect DICTrank, DILIrank and DIRIL.
6. Generate leakage-safe out-of-fold predictions.
7. Test external agreement, model agreement and molecular similarity as reliability signals.
8. Train reliability-weighted models.
9. Run ablations.
10. Evaluate calibration and robustness.


In [16]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score


# ============================================================
# 1. EXACT UniTox toxicity column prefixes
# ============================================================

toxicity_prefixes = {
    "Cardio": "cardio_toxicity",
    "Dermatological": "dermatological_toxicity",
    "Hematological": "hematological",
    "Infertility": "infertility",
    "Liver": "liver_toxicity",
    "Ototoxicity": "ototoxicity",
    "Pulmonary": "pulmonary_toxicity",
    "Renal": "renal_toxicity"
}


# ============================================================
# 2. Common RF + CV configuration
# ============================================================

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

rf_params = {
    "n_estimators": 500,
    "random_state": 0,
    "n_jobs": -1
}


# ============================================================
# 3. Evaluation function
# ============================================================

def evaluate_target(target, X, y, label_type):

    rf = RandomForestClassifier(**rf_params)

    scores = cross_val_score(
        rf,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    return {
        "target": target,
        "label_type": label_type,
        "n_samples": len(y),
        "positive": int(np.sum(y == 1)),
        "negative": int(np.sum(y == 0)),
        "mean_auc": scores.mean(),
        "std_auc": scores.std(),
        "fold_scores": scores
    }


# ============================================================
# 4. ALL BINARY LABELS
# ============================================================

binary_results = []

print("=" * 70)
print("ALL BINARY LABEL EXPERIMENTS")
print("=" * 70)

for toxicity_name, prefix in toxicity_prefixes.items():

    target = f"{prefix}_binary_rating_0_1"

    # Check that the column actually exists
    if target not in df.columns:
        print(f"\nWARNING: Missing column: {target}")
        continue

    y = df[target].astype(int).to_numpy()

    print(f"\nRunning: {toxicity_name}")
    print(f"Target: {target}")
    print(f"Samples: {len(y)}")
    print(f"Class distribution: {np.bincount(y)}")

    result = evaluate_target(
        target=target,
        X=morgan,
        y=y,
        label_type="All Binary"
    )

    binary_results.append(result)

    print(
        f"AUC = {result['mean_auc']:.4f} "
        f"+/- {result['std_auc']:.4f}"
    )


# ============================================================
# 5. CONFIDENT LABELS ONLY
# ============================================================

confident_results = []

print("\n")
print("=" * 70)
print("CONFIDENT LABEL EXPERIMENTS")
print("=" * 70)

for toxicity_name, prefix in toxicity_prefixes.items():

    target = f"{prefix}_confident_ternary_rating_0_1"

    if target not in df.columns:
        print(f"\nWARNING: Missing column: {target}")
        continue

    mask = df[target].notna()

    X_conf = morgan[mask]

    y_conf = (
        df.loc[mask, target]
        .astype(int)
        .to_numpy()
    )

    print(f"\nRunning: {toxicity_name}")
    print(f"Target: {target}")
    print(f"Confident samples: {len(y_conf)}")
    print(f"Removed: {len(df) - len(y_conf)}")
    print(f"Class distribution: {np.bincount(y_conf)}")

    result = evaluate_target(
        target=target,
        X=X_conf,
        y=y_conf,
        label_type="Confident Only"
    )

    confident_results.append(result)

    print(
        f"AUC = {result['mean_auc']:.4f} "
        f"+/- {result['std_auc']:.4f}"
    )


# ============================================================
# 6. CREATE SUMMARY DATAFRAMES
# ============================================================

binary_summary = pd.DataFrame([
    {
        "Toxicity": toxicity_name,
        "Samples": r["n_samples"],
        "Positive": r["positive"],
        "Negative": r["negative"],
        "Mean AUC": r["mean_auc"],
        "Std AUC": r["std_auc"]
    }
    for toxicity_name, r in zip(
        [x for x in toxicity_prefixes if
         f"{toxicity_prefixes[x]}_binary_rating_0_1" in df.columns],
        binary_results
    )
])


confident_summary = pd.DataFrame([
    {
        "Toxicity": toxicity_name,
        "Samples": r["n_samples"],
        "Positive": r["positive"],
        "Negative": r["negative"],
        "Mean AUC": r["mean_auc"],
        "Std AUC": r["std_auc"]
    }
    for toxicity_name, r in zip(
        [x for x in toxicity_prefixes if
         f"{toxicity_prefixes[x]}_confident_ternary_rating_0_1"
         in df.columns],
        confident_results
    )
])


# ============================================================
# 7. MERGE RESULTS
# ============================================================

comparison = binary_summary[
    ["Toxicity", "Samples", "Mean AUC", "Std AUC"]
].copy()

comparison = comparison.rename(columns={
    "Samples": "All Samples",
    "Mean AUC": "All Binary AUC",
    "Std AUC": "All Binary Std"
})

comparison["Confident Samples"] = confident_summary["Samples"]

comparison["Confident AUC"] = confident_summary["Mean AUC"]

comparison["Confident Std"] = confident_summary["Std AUC"]

comparison["AUC Difference"] = (
    comparison["Confident AUC"]
    - comparison["All Binary AUC"]
)

comparison["Removed %"] = (
    (comparison["All Samples"]
     - comparison["Confident Samples"])
    / comparison["All Samples"]
    * 100
)


# ============================================================
# 8. DISPLAY
# ============================================================

print("\n")
print("=" * 100)
print("ALL LABELS vs CONFIDENT LABELS")
print("=" * 100)

display(
    comparison.round(4)
)


# ============================================================
# 9. SAVE
# ============================================================

comparison.to_csv(
    "UniTox_RF_Morgan_All_vs_Confident.csv",
    index=False
)

print("\nResults saved successfully.")

ALL BINARY LABEL EXPERIMENTS

Running: Cardio
Target: cardio_toxicity_binary_rating_0_1
Samples: 1349
Class distribution: [594 755]
AUC = 0.7701 +/- 0.0310

Running: Dermatological
Target: dermatological_toxicity_binary_rating_0_1
Samples: 1349
Class distribution: [ 150 1199]
AUC = 0.7032 +/- 0.0777

Running: Hematological
Target: hematological_binary_rating_0_1
Samples: 1349
Class distribution: [473 876]
AUC = 0.7788 +/- 0.0383

Running: Infertility
Target: infertility_binary_rating_0_1
Samples: 1349
Class distribution: [846 503]
AUC = 0.7837 +/- 0.0369

Running: Liver
Target: liver_toxicity_binary_rating_0_1
Samples: 1349
Class distribution: [613 736]
AUC = 0.7698 +/- 0.0410

Running: Ototoxicity
Target: ototoxicity_binary_rating_0_1
Samples: 1349
Class distribution: [1228  121]
AUC = 0.6790 +/- 0.0963

Running: Pulmonary
Target: pulmonary_toxicity_binary_rating_0_1
Samples: 1349
Class distribution: [851 498]
AUC = 0.7367 +/- 0.0523

Running: Renal
Target: renal_toxicity_binary_ratin

,Toxicity,All Samples,All Binary AUC,All Binary Std,Confident Samples,Confident AUC,Confident Std,AUC Difference,Removed %
0,Cardio,1349,0.7701,0.0310,901,0.8030,0.0570,0.0329,33.2098
1,Dermatological,1349,0.7032,0.0777,955,0.7321,0.0635,0.0289,29.2068
2,Hematological,1349,0.7788,0.0383,1075,0.8221,0.0297,0.0433,20.3113
3,Infertility,1349,0.7837,0.0369,1021,0.8274,0.0321,0.0437,24.3143
4,Liver,1349,0.7698,0.0410,878,0.8306,0.0365,0.0608,34.9148
5,Ototoxicity,1349,0.6790,0.0963,1183,0.7741,0.1300,0.0951,12.3054
6,Pulmonary,1349,0.7367,0.0523,997,0.7603,0.0484,0.0236,26.0934
7,Renal,1349,0.7616,0.0265,948,0.8037,0.0296,0.0421,29.7257



Results saved successfully.


In [17]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
import pandas as pd
import numpy as np


def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol,
        includeChirality=False
    )


df["scaffold"] = df["smiles"].apply(get_scaffold)

print("Total molecules:", len(df))
print("Valid scaffolds:", df["scaffold"].notna().sum())
print("Unique scaffolds:", df["scaffold"].nunique())

print("\nTop 20 scaffold frequencies:")
print(
    df["scaffold"]
    .value_counts()
    .head(20)
)

print("\nMolecules without scaffold:")
print(df["scaffold"].isna().sum())

[16:55:22] WARNING: not removing hydrogen atom without neighbors
[16:55:22] WARNING: not removing hydrogen atom without neighbors
[16:55:22] WARNING: not removing hydrogen atom without neighbors
[16:55:22] WARNING: not removing hydrogen atom without neighbors
[16:55:22] WARNING: not removing hydrogen atom without neighbors
[16:55:22] WARNING: not removing hydrogen atom without neighbors
[16:55:22] WARNING: not removing hydrogen atom without neighbors
[16:55:22] WARNING: not removing hydrogen atom without neighbors
[16:55:22] WARNING: not removing hydrogen atom without neighbors
[16:55:22] WARNING: not removing hydrogen atom without neighbors


Total molecules: 1349
Valid scaffolds: 1349
Unique scaffolds: 802

Top 20 scaffold frequencies:
scaffold
                                      122
c1ccccc1                               77
O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21        15
O=C1C=C2CCC3C4CCCC4CCC3C2CC1           12
O=C1C=CC2C(=O)C3=Cc4ccccc4CC3CC2C1     10
c1ccncc1                                8
c1ccccc1.c1ccccc1                       8
c1ccc(Cc2ccccc2)cc1                     8
c1ccc2c(c1)Sc1ccccc1N2CCCN1CCNCC1       8
c1c[nH]cn1                              7
O=C(c1ccccc1)c1ccccc1                   7
O=c1[nH]cnc2[nH]cnc12                   6
c1ncc2ncn(C3CCCO3)c2n1                  6
O=C1CN=C(c2ccccc2)c2ccccc2N1            6
c1ccc(CC2CCCCC2)cc1                     6
c1ncc2nc[nH]c2n1                        5
O=C(Cc1ccccc1)NC1C(=O)N2CCSC12          5
C1CCC2C(C1)CCC1C3CCCC3CCC21             5
c1ccc(Oc2ccccc2)cc1                     5
O=C(Cc1ccccc1)NC1C(=O)N2C=CCSC12        5
Name: count, dtype: int64

Molecules without scaffold:


[16:55:24] WARNING: not removing hydrogen atom without neighbors
[16:55:24] WARNING: not removing hydrogen atom without neighbors


In [18]:
from collections import defaultdict
import numpy as np
import pandas as pd


# ============================================================
# Scaffold groups
# ============================================================

scaffold_groups = defaultdict(list)

for idx, scaffold in enumerate(df["scaffold"]):
    scaffold_groups[scaffold].append(idx)

# Convert to lists
scaffold_groups = {
    scaffold: np.array(indices)
    for scaffold, indices in scaffold_groups.items()
}

print("Number of scaffold groups:", len(scaffold_groups))

print("\nLargest scaffold groups:")
largest_groups = sorted(
    scaffold_groups.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:20]

for i, (scaffold, indices) in enumerate(largest_groups, 1):
    display_scaffold = "<EMPTY>" if scaffold == "" else scaffold

    print(
        f"{i:2d}. "
        f"size={len(indices):3d} | "
        f"{display_scaffold}"
    )


# ============================================================
# Create scaffold-balanced folds
# ============================================================

n_folds = 10
rng = np.random.RandomState(0)

# Shuffle scaffold groups first so ties don't depend on
# dictionary ordering.
groups = list(scaffold_groups.items())
rng.shuffle(groups)

# Sort largest groups first
groups = sorted(
    groups,
    key=lambda x: len(x[1]),
    reverse=True
)

fold_indices = [[] for _ in range(n_folds)]
fold_sizes = np.zeros(n_folds, dtype=int)

# Greedy balancing:
# assign each new scaffold to the currently smallest fold
for scaffold, indices in groups:

    fold_id = np.argmin(fold_sizes)

    fold_indices[fold_id].extend(indices.tolist())
    fold_sizes[fold_id] += len(indices)


# Convert to numpy arrays
fold_indices = [
    np.array(sorted(indices), dtype=int)
    for indices in fold_indices
]


# ============================================================
# Basic fold information
# ============================================================

print("\n" + "=" * 70)
print("SCAFFOLD-BALANCED FOLD SIZES")
print("=" * 70)

for i, indices in enumerate(fold_indices):

    print(
        f"Fold {i + 1}: "
        f"{len(indices)} molecules | "
        f"{len(set(df.loc[indices, 'scaffold']))} scaffolds"
    )

print(
    "\nTotal assigned:",
    sum(len(x) for x in fold_indices)
)

print(
    "Original molecules:",
    len(df)
)


# ============================================================
# VERIFY every molecule appears exactly once
# ============================================================

all_test_indices = np.concatenate(fold_indices)

print(
    "\nUnique assigned molecules:",
    len(np.unique(all_test_indices))
)

print(
    "Expected:",
    len(df)
)


# ============================================================
# VERIFY NO SCAFFOLD LEAKAGE
# ============================================================

print("\n" + "=" * 70)
print("SCAFFOLD LEAKAGE CHECK")
print("=" * 70)

leakage_found = False

for test_fold in range(n_folds):

    test_idx = fold_indices[test_fold]

    train_idx = np.concatenate([
        fold_indices[j]
        for j in range(n_folds)
        if j != test_fold
    ])

    train_scaffolds = set(
        df.loc[train_idx, "scaffold"]
    )

    test_scaffolds = set(
        df.loc[test_idx, "scaffold"]
    )

    overlap = train_scaffolds.intersection(
        test_scaffolds
    )

    print(
        f"Fold {test_fold + 1}: "
        f"train={len(train_idx)}, "
        f"test={len(test_idx)}, "
        f"scaffold overlap={len(overlap)}"
    )

    if len(overlap) > 0:
        leakage_found = True


print("\nLeakage detected:", leakage_found)

Number of scaffold groups: 802

Largest scaffold groups:
 1. size=122 | <EMPTY>
 2. size= 77 | c1ccccc1
 3. size= 15 | O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21
 4. size= 12 | O=C1C=C2CCC3C4CCCC4CCC3C2CC1
 5. size= 10 | O=C1C=CC2C(=O)C3=Cc4ccccc4CC3CC2C1
 6. size=  8 | c1ccncc1
 7. size=  8 | c1ccccc1.c1ccccc1
 8. size=  8 | c1ccc(Cc2ccccc2)cc1
 9. size=  8 | c1ccc2c(c1)Sc1ccccc1N2CCCN1CCNCC1
10. size=  7 | c1c[nH]cn1
11. size=  7 | O=C(c1ccccc1)c1ccccc1
12. size=  6 | O=c1[nH]cnc2[nH]cnc12
13. size=  6 | c1ncc2ncn(C3CCCO3)c2n1
14. size=  6 | O=C1CN=C(c2ccccc2)c2ccccc2N1
15. size=  6 | c1ccc(CC2CCCCC2)cc1
16. size=  5 | c1ncc2nc[nH]c2n1
17. size=  5 | O=C(Cc1ccccc1)NC1C(=O)N2CCSC12
18. size=  5 | C1CCC2C(C1)CCC1C3CCCC3CCC21
19. size=  5 | c1ccc(Oc2ccccc2)cc1
20. size=  5 | O=C(Cc1ccccc1)NC1C(=O)N2C=CCSC12

SCAFFOLD-BALANCED FOLD SIZES
Fold 1: 135 molecules | 14 scaffolds
Fold 2: 135 molecules | 59 scaffolds
Fold 3: 135 molecules | 90 scaffolds
Fold 4: 135 molecules | 91 scaffolds
Fold 5: 135 mol

In [19]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import numpy as np

# ---------------------------------------------------------
# Generate Morgan fingerprints
# ---------------------------------------------------------
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

X_morgan = np.array([
    morgan_generator.GetFingerprintAsNumPy(Chem.MolFromSmiles(smiles))
    for smiles in df["smiles"]
])

print("Morgan fingerprint shape:", X_morgan.shape)

[16:55:25] WARNING: not removing hydrogen atom without neighbors
[16:55:25] WARNING: not removing hydrogen atom without neighbors
[16:55:25] WARNING: not removing hydrogen atom without neighbors
[16:55:25] WARNING: not removing hydrogen atom without neighbors
[16:55:25] WARNING: not removing hydrogen atom without neighbors
[16:55:25] WARNING: not removing hydrogen atom without neighbors
[16:55:25] WARNING: not removing hydrogen atom without neighbors
[16:55:25] WARNING: not removing hydrogen atom without neighbors
[16:55:25] WARNING: not removing hydrogen atom without neighbors
[16:55:25] WARNING: not removing hydrogen atom without neighbors


Morgan fingerprint shape: (1349, 2048)


[16:55:25] WARNING: not removing hydrogen atom without neighbors
[16:55:25] WARNING: not removing hydrogen atom without neighbors


In [20]:
# Show all columns containing "liver"
[c for c in df.columns if "liver" in c.lower()]

['liver_toxicity_reasoning',
 'liver_toxicity_ternary_rating',
 'liver_toxicity_binary_rating',
 'liver_toxicity_confident_ternary_rating_0_1',
 'liver_toxicity_binary_rating_0_1']

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

# ---------------------------------------------------------
# Liver binary target
# ---------------------------------------------------------
target_col = "liver_toxicity_binary_rating_0_1"

X = X_morgan
y = df[target_col].values.astype(int)

print("Target:", target_col)
print("Class distribution:", np.bincount(y))

# ---------------------------------------------------------
# Scaffold-balanced 10-fold evaluation
# ---------------------------------------------------------
fold_aucs = []

for fold in range(10):

    test_idx = fold_indices[fold]

    train_idx = np.concatenate([
        fold_indices[j]
        for j in range(10)
        if j != fold
    ])

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    model = RandomForestClassifier(
        n_estimators=500,
        random_state=0,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_prob)
    fold_aucs.append(auc)

    print(
        f"Fold {fold+1}: "
        f"train={len(train_idx)}, "
        f"test={len(test_idx)}, "
        f"AUC={auc:.4f}"
    )

print("\n" + "="*60)
print("SCAFFOLD-BALANCED RF + MORGAN — LIVER")
print("="*60)

print("Fold AUCs:", [round(x, 4) for x in fold_aucs])
print(f"Mean AUC: {np.mean(fold_aucs):.4f}")
print(f"Std AUC : {np.std(fold_aucs):.4f}")

Target: liver_toxicity_binary_rating_0_1
Class distribution: [613 736]
Fold 1: train=1214, test=135, AUC=0.5541
Fold 2: train=1214, test=135, AUC=0.6697
Fold 3: train=1214, test=135, AUC=0.7705
Fold 4: train=1214, test=135, AUC=0.7004
Fold 5: train=1214, test=135, AUC=0.7666
Fold 6: train=1214, test=135, AUC=0.7198
Fold 7: train=1214, test=135, AUC=0.7582
Fold 8: train=1214, test=135, AUC=0.6447
Fold 9: train=1214, test=135, AUC=0.7627
Fold 10: train=1215, test=134, AUC=0.6242

SCAFFOLD-BALANCED RF + MORGAN — LIVER
Fold AUCs: [0.5541, 0.6697, 0.7705, 0.7004, 0.7666, 0.7198, 0.7582, 0.6447, 0.7627, 0.6242]
Mean AUC: 0.6971
Std AUC : 0.0693


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

# =========================================================
# EXACT TARGET COLUMNS FROM YOUR DATAFRAME
# =========================================================
toxicity_targets = {
    "Cardiotoxicity": "cardio_toxicity_binary_rating_0_1",
    "Dermatological": "dermatological_toxicity_binary_rating_0_1",
    "Hematological": "hematological_binary_rating_0_1",
    "Infertility": "infertility_binary_rating_0_1",
    "Liver": "liver_toxicity_binary_rating_0_1",
    "Ototoxicity": "ototoxicity_binary_rating_0_1",
    "Pulmonary": "pulmonary_toxicity_binary_rating_0_1",
    "Renal": "renal_toxicity_binary_rating_0_1"
}

results = []

# =========================================================
# SCAFFOLD-BALANCED 10-FOLD RF + MORGAN
# =========================================================

for toxicity, target_col in toxicity_targets.items():

    y = df[target_col].values.astype(int)

    fold_aucs = []

    for fold in range(10):

        test_idx = fold_indices[fold]

        train_idx = np.concatenate([
            fold_indices[j]
            for j in range(10)
            if j != fold
        ])

        X_train = X_morgan[train_idx]
        X_test = X_morgan[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        model = RandomForestClassifier(
            n_estimators=500,
            random_state=0,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        y_prob = model.predict_proba(X_test)[:, 1]

        auc = roc_auc_score(y_test, y_prob)
        fold_aucs.append(auc)

    mean_auc = np.mean(fold_aucs)
    std_auc = np.std(fold_aucs)

    results.append({
        "Toxicity": toxicity,
        "Mean AUC": mean_auc,
        "Std AUC": std_auc
    })

    print(
        f"{toxicity:16s} | "
        f"AUC = {mean_auc:.4f} ± {std_auc:.4f}"
    )

# =========================================================
# FINAL RESULTS TABLE
# =========================================================

scaffold_results = pd.DataFrame(results)

print("\n" + "=" * 70)
print("SCAFFOLD-BALANCED RF + MORGAN — ALL 8 TOXICITY TASKS")
print("=" * 70)

display(
    scaffold_results.style.format({
        "Mean AUC": "{:.4f}",
        "Std AUC": "{:.4f}"
    })
)

Cardiotoxicity   | AUC = 0.6934 ± 0.0846
Dermatological   | AUC = 0.5804 ± 0.1105
Hematological    | AUC = 0.7096 ± 0.0844
Infertility      | AUC = 0.6719 ± 0.0799
Liver            | AUC = 0.6971 ± 0.0693
Ototoxicity      | AUC = 0.6162 ± 0.1028
Pulmonary        | AUC = 0.6679 ± 0.0781
Renal            | AUC = 0.7032 ± 0.0828

SCAFFOLD-BALANCED RF + MORGAN — ALL 8 TOXICITY TASKS


,Toxicity,Mean AUC,Std AUC
0,Cardiotoxicity,0.6934,0.0846
1,Dermatological,0.5804,0.1105
2,Hematological,0.7096,0.0844
3,Infertility,0.6719,0.0799
4,Liver,0.6971,0.0693
5,Ototoxicity,0.6162,0.1028
6,Pulmonary,0.6679,0.0781
7,Renal,0.7032,0.0828


In [26]:
import pandas as pd
from pathlib import Path

# =========================================================
# FIND THE THREE EXTERNAL VALIDATION FILES
# =========================================================

BASE_DIR = Path.cwd()

print("Notebook working directory:")
print(BASE_DIR)

# ---------------------------------------------------------
# File paths
# ---------------------------------------------------------

dictrank_path = BASE_DIR / "dictrank_raw.csv"
dilirank_path = BASE_DIR / "dilirank_raw.xlsx"
diril_path = BASE_DIR / "A. DIRIL (317)-Table 1.csv"

# Check that all files exist
print("\nFile check:")
print("DICTrank:", dictrank_path.exists(), dictrank_path)
print("DILIrank:", dilirank_path.exists(), dilirank_path)
print("DIRIL:", diril_path.exists(), diril_path)

# =========================================================
# LOAD DATASETS
# =========================================================

if not dictrank_path.exists():
    raise FileNotFoundError(
        f"DICTrank not found: {dictrank_path}\n"
        "Download dictrank_raw.csv and place it beside this notebook."
    )

if not dilirank_path.exists():
    raise FileNotFoundError(
        f"DILIrank not found: {dilirank_path}\n"
        "Download dilirank_raw.xlsx and place it beside this notebook."
    )

if not diril_path.exists():
    raise FileNotFoundError(
        f"DIRIL not found: {diril_path}\n"
        "Download A. DIRIL (317)-Table 1.csv and place it beside this notebook."
    )

# =========================================================
# READ FILES
# =========================================================

dictrank = pd.read_csv(dictrank_path)

dilirank = pd.read_excel(
    dilirank_path,
    sheet_name="supporting table 1"
)

diril = pd.read_csv(diril_path)

# =========================================================
# AUDIT
# =========================================================

print("\n" + "=" * 70)
print("EXTERNAL DATASET AUDIT")
print("=" * 70)

print("\nDICTrank")
print("-" * 70)
print("Shape:", dictrank.shape)
print("Columns:")
print(dictrank.columns.tolist())

print("\nDILIrank")
print("-" * 70)
print("Shape:", dilirank.shape)
print("Columns:")
print(dilirank.columns.tolist())

print("\nDIRIL")
print("-" * 70)
print("Shape:", diril.shape)
print("Columns:")
print(diril.columns.tolist())

Notebook working directory:
C:\Users\DEEPAJYOTI

File check:
DICTrank: True C:\Users\DEEPAJYOTI\dictrank_raw.csv
DILIrank: True C:\Users\DEEPAJYOTI\dilirank_raw.xlsx
DIRIL: True C:\Users\DEEPAJYOTI\A. DIRIL (317)-Table 1.csv

EXTERNAL DATASET AUDIT

DICTrank
----------------------------------------------------------------------
Shape: (1318, 10)
Columns:
['Unnamed: 0', 'Trade Name', 'Generic/Proper Name(s)', 'Active Ingredient(s)', 'Cardiotoxicity', 'Label Section ', 'DICT _ Concern', 'Keywords', 'DIC Severity Level', 'split']

DILIrank
----------------------------------------------------------------------
Shape: (1036, 16)
Columns:
['LTKBID', 'PubChem_CID', 'LabelCompoundName', 'ApprovalYear', 'DILIConcern', 'SeverityClass', 'LabelSection', 'AdjudicatedDILI', 'vDILIConcern', 'Greene_Annotation', 'Sakatis_Annotation', 'Xu_Annotation', 'Zhu_Annotation', 'SMILES', 'URL (accessed on 1/21/2016)', 'VER_DISP (1=LTKB-BD, 2=others)']

DIRIL
-----------------------------------------------------

In [25]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [27]:
# =========================================================
# DILIrank LABEL AUDIT
# =========================================================

print("=" * 70)
print("DILIrank — DILIConcern")
print("=" * 70)

print(dilirank["DILIConcern"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DILIrank — AdjudicatedDILI")
print("=" * 70)

print(dilirank["AdjudicatedDILI"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DILIrank — SeverityClass")
print("=" * 70)

print(dilirank["SeverityClass"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("Examples")
print("=" * 70)

display(
    dilirank[
        [
            "LabelCompoundName",
            "DILIConcern",
            "SeverityClass",
            "AdjudicatedDILI",
            "SMILES"
        ]
    ].head(20)
)

DILIrank — DILIConcern
DILIConcern
Less-DILI-Concern    488
No-DILI-Concern      321
Most-DILI-Concern    208
No-DILI-concern       18
Most-DILI-concern      1
Name: count, dtype: int64

DILIrank — AdjudicatedDILI
AdjudicatedDILI
NaN                625
Yes                399
Yes(literature)     12
Name: count, dtype: int64

DILIrank — SeverityClass
SeverityClass
0    338
3    309
8    159
5     62
7     60
4     53
2     46
6      5
1      4
Name: count, dtype: int64

Examples


,LabelCompoundName,DILIConcern,SeverityClass,AdjudicatedDILI,SMILES
0,ceftriaxone,Less-DILI-Concern,4,Yes,CN1C(=NC(=O)C(=O)N1)SCC2=C(N3[C@@H]([C@@H](C3=...
1,trimethoprim,Less-DILI-Concern,4,Yes,COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N
2,tetracycline,Less-DILI-Concern,2,Yes,CC1(C2CC3C(C(=O)/C(=C(\N)/O)/C(=O)C3(C(=O)C2=C...
3,dapsone,Less-DILI-Concern,3,Yes,C1=CC(=CC=C1N)S(=O)(=O)C2=CC=C(C=C2)N
4,pyrazinamide,Less-DILI-Concern,3,Yes,C1=CN=C(C=N1)C(=O)N
5,fenofibrate,Less-DILI-Concern,3,Yes,CC(C)OC(=O)C(C)(C)OC1=CC=C(C=C1)C(=O)C2=CC=C(C...
6,cyclophosphamide,Less-DILI-Concern,5,Yes,C1CNP(=O)(OC1)N(CCCl)CCCl
7,chlorpromazine,Less-DILI-Concern,2,Yes,CN(C)CCCN1C2=CC=CC=C2SC3=C1C=C(C=C3)Cl
8,doxorubicin,Less-DILI-Concern,3,Yes,C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC...
9,clindamycin,Less-DILI-Concern,3,Yes,CCC[C@@H]1C[C@H](N(C1)C)C(=O)NC([C@@H]2[C@@H](...


In [28]:
# =========================================================
# UNITOX LIVER LABEL AUDIT
# =========================================================

print("=" * 70)
print("UniTox liver binary labels")
print("=" * 70)

print(
    df["liver_toxicity_binary_rating_0_1"]
    .value_counts(dropna=False)
)

print("\nRelevant UniTox columns:")

for c in df.columns:
    if "liver" in c.lower():
        print(" -", c)

UniTox liver binary labels
liver_toxicity_binary_rating_0_1
1    736
0    613
Name: count, dtype: int64

Relevant UniTox columns:
 - liver_toxicity_reasoning
 - liver_toxicity_ternary_rating
 - liver_toxicity_binary_rating
 - liver_toxicity_confident_ternary_rating_0_1
 - liver_toxicity_binary_rating_0_1


In [29]:
print("\nPotential drug identity columns:")

for c in df.columns:
    c_lower = c.lower()

    if any(
        keyword in c_lower
        for keyword in [
            "drug",
            "name",
            "compound",
            "smiles",
            "pubchem",
            "inchikey",
            "drugbank"
        ]
    ):
        print(" -", c)


Potential drug identity columns:
 - generic_name
 - smiles
 - all_smiles


In [30]:
# =========================================================
# DILIrank LABEL AUDIT
# =========================================================

print("=" * 70)
print("DILIrank — DILIConcern")
print("=" * 70)
print(dilirank["DILIConcern"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DILIrank — AdjudicatedDILI")
print("=" * 70)
print(dilirank["AdjudicatedDILI"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DILIrank — SeverityClass")
print("=" * 70)
print(dilirank["SeverityClass"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DILIrank — Example records")
print("=" * 70)

display(
    dilirank[
        [
            "LabelCompoundName",
            "DILIConcern",
            "SeverityClass",
            "AdjudicatedDILI",
            "SMILES"
        ]
    ].head(20)
)

DILIrank — DILIConcern
DILIConcern
Less-DILI-Concern    488
No-DILI-Concern      321
Most-DILI-Concern    208
No-DILI-concern       18
Most-DILI-concern      1
Name: count, dtype: int64

DILIrank — AdjudicatedDILI
AdjudicatedDILI
NaN                625
Yes                399
Yes(literature)     12
Name: count, dtype: int64

DILIrank — SeverityClass
SeverityClass
0    338
3    309
8    159
5     62
7     60
4     53
2     46
6      5
1      4
Name: count, dtype: int64

DILIrank — Example records


,LabelCompoundName,DILIConcern,SeverityClass,AdjudicatedDILI,SMILES
0,ceftriaxone,Less-DILI-Concern,4,Yes,CN1C(=NC(=O)C(=O)N1)SCC2=C(N3[C@@H]([C@@H](C3=...
1,trimethoprim,Less-DILI-Concern,4,Yes,COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N
2,tetracycline,Less-DILI-Concern,2,Yes,CC1(C2CC3C(C(=O)/C(=C(\N)/O)/C(=O)C3(C(=O)C2=C...
3,dapsone,Less-DILI-Concern,3,Yes,C1=CC(=CC=C1N)S(=O)(=O)C2=CC=C(C=C2)N
4,pyrazinamide,Less-DILI-Concern,3,Yes,C1=CN=C(C=N1)C(=O)N
5,fenofibrate,Less-DILI-Concern,3,Yes,CC(C)OC(=O)C(C)(C)OC1=CC=C(C=C1)C(=O)C2=CC=C(C...
6,cyclophosphamide,Less-DILI-Concern,5,Yes,C1CNP(=O)(OC1)N(CCCl)CCCl
7,chlorpromazine,Less-DILI-Concern,2,Yes,CN(C)CCCN1C2=CC=CC=C2SC3=C1C=C(C=C3)Cl
8,doxorubicin,Less-DILI-Concern,3,Yes,C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC...
9,clindamycin,Less-DILI-Concern,3,Yes,CCC[C@@H]1C[C@H](N(C1)C)C(=O)NC([C@@H]2[C@@H](...


In [31]:
# =========================================================
# 1. DILIrank LABEL DISTRIBUTIONS
# =========================================================

for col in ["DILIConcern", "SeverityClass", "AdjudicatedDILI"]:
    print("\n" + "=" * 60)
    print(f"{col} — value counts")
    print("=" * 60)
    print(dilirank[col].value_counts(dropna=False).to_string())


# =========================================================
# 2. CROSS-TABULATE DILI CONCERN AND ADJUDICATION
# =========================================================

print("\nDILIConcern vs AdjudicatedDILI")

display(
    pd.crosstab(
        dilirank["DILIConcern"],
        dilirank["AdjudicatedDILI"],
        margins=True,
        dropna=False
    )
)


# =========================================================
# 3. UNITOX DRUG IDENTITY QUALITY
# =========================================================

print("\n" + "=" * 60)
print("UniTox identity audit")
print("=" * 60)

print("Rows:", len(df))
print("Unique generic names:", df["generic_name"].nunique())
print("Missing generic names:", df["generic_name"].isna().sum())
print("Missing SMILES:", df["smiles"].isna().sum())
print("Missing all_smiles:", df["all_smiles"].isna().sum())

display(
    df[
        [
            "generic_name",
            "smiles",
            "liver_toxicity_binary_rating_0_1"
        ]
    ].head(15)
)


DILIConcern — value counts
DILIConcern
Less-DILI-Concern    488
No-DILI-Concern      321
Most-DILI-Concern    208
No-DILI-concern       18
Most-DILI-concern      1

SeverityClass — value counts
SeverityClass
0    338
3    309
8    159
5     62
7     60
4     53
2     46
6      5
1      4

AdjudicatedDILI — value counts
AdjudicatedDILI
NaN                625
Yes                399
Yes(literature)     12

DILIConcern vs AdjudicatedDILI


AdjudicatedDILI,Yes,Yes(literature),NaN,All
DILIConcern,,,,
Less-DILI-Concern,249,2,237,488
Most-DILI-Concern,124,9,75,208
Most-DILI-concern,0,0,1,1
No-DILI-Concern,26,1,294,321
No-DILI-concern,0,0,18,18
All,399,12,625,1036



UniTox identity audit
Rows: 1349
Unique generic names: 1349
Missing generic names: 0
Missing SMILES: 0
Missing all_smiles: 0


,generic_name,smiles,liver_toxicity_binary_rating_0_1
0,68GA-DOTATATE,CC(O)C(NC(=O)C1CSSCC(NC(=O)C(Cc2ccccc2)NC(=O)C...,0
1,ABACAVIR,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,1
2,ABALOPARATIDE,CCC(C)C(NC(=O)C(CO)NC(=O)C(CCCCN)NC(=O)CNC(=O)...,0
3,ABEMACICLIB,CCN1CCN(Cc2ccc(Nc3ncc(F)c(-c4cc(F)c5nc(C)n(C(C...,1
4,ABIRATERONE ACETATE,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,1
5,ACALABRUTINIB,CC#CC(=O)N1CCCC1c1nc(-c2ccc(C(=O)Nc3ccccn3)cc2...,1
6,ACARBOSE,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,1
7,ACEBUTOLOL HYDROCHLORIDE,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1.Cl,1
8,ACETAZOLAMIDE,CC(=O)Nc1nnc(S(N)(=O)=O)s1,1
9,ACETOHYDROXAMIC ACID,CC(=O)NO,0


In [32]:
import pandas as pd
import re
import numpy as np

# =========================================================
# NORMALIZE DRUG NAMES
# =========================================================

def normalize_drug_name(value):
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    # Normalize punctuation and whitespace
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value if value else None


# Work on copies to preserve the original datasets
unitox_match = df[
    ["generic_name", "smiles", "liver_toxicity_binary_rating_0_1"]
].copy()

dilirank_match = dilirank[
    ["LabelCompoundName", "SMILES", "DILIConcern",
     "SeverityClass", "AdjudicatedDILI"]
].copy()

unitox_match["normalized_name"] = (
    unitox_match["generic_name"].apply(normalize_drug_name)
)

dilirank_match["normalized_name"] = (
    dilirank_match["LabelCompoundName"].apply(normalize_drug_name)
)

# =========================================================
# CHECK DUPLICATE KEYS BEFORE MERGING
# =========================================================

print("UniTox rows:", len(unitox_match))
print("DILIrank rows:", len(dilirank_match))

print(
    "UniTox duplicate normalized names:",
    unitox_match["normalized_name"].duplicated().sum()
)

print(
    "DILIrank duplicate normalized names:",
    dilirank_match["normalized_name"].duplicated().sum()
)

# Keep only records with usable normalized names
unitox_valid_names = unitox_match.dropna(
    subset=["normalized_name"]
).copy()

dilirank_valid_names = dilirank_match.dropna(
    subset=["normalized_name"]
).copy()

# Identify names occurring in both datasets
unitox_names = set(unitox_valid_names["normalized_name"])
dilirank_names = set(dilirank_valid_names["normalized_name"])

common_names = unitox_names.intersection(dilirank_names)

print("\nUnique normalized UniTox names:", len(unitox_names))
print("Unique normalized DILIrank names:", len(dilirank_names))
print("Common normalized names:", len(common_names))

# =========================================================
# SHOW CANDIDATE MATCHES
# =========================================================

candidate_matches = unitox_valid_names[
    unitox_valid_names["normalized_name"].isin(common_names)
].merge(
    dilirank_valid_names,
    on="normalized_name",
    how="inner",
    suffixes=("_unitox", "_dilirank"),
    validate="many_to_many"
)

print("\nCandidate matched rows:", len(candidate_matches))
print(
    "Unique UniTox drug names matched:",
    candidate_matches["normalized_name"].nunique()
)

display(
    candidate_matches[
        [
            "generic_name",
            "LabelCompoundName",
            "normalized_name",
            "smiles",
            "SMILES",
            "liver_toxicity_binary_rating_0_1",
            "DILIConcern",
            "SeverityClass",
            "AdjudicatedDILI"
        ]
    ].head(30)
)

UniTox rows: 1349
DILIrank rows: 1036
UniTox duplicate normalized names: 1
DILIrank duplicate normalized names: 0

Unique normalized UniTox names: 1348
Unique normalized DILIrank names: 1036
Common normalized names: 497

Candidate matched rows: 497
Unique UniTox drug names matched: 497


,generic_name,LabelCompoundName,normalized_name,smiles,SMILES,liver_toxicity_binary_rating_0_1,DILIConcern,SeverityClass,AdjudicatedDILI
0,ABACAVIR,abacavir,abacavir,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,C1CC1NC2=NC(=NC3=C2N=CN3C4C[C@H](C=C4)CO)N,1,Most-DILI-Concern,8,Yes
1,ACARBOSE,acarbose,acarbose,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,C[C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O[C@@H]2[C...,1,Most-DILI-Concern,8,Yes
2,ACETAZOLAMIDE,acetazolamide,acetazolamide,CC(=O)Nc1nnc(S(N)(=O)=O)s1,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,1,Most-DILI-Concern,8,Yes
3,ACETOHYDROXAMIC ACID,acetohydroxamic acid,acetohydroxamic acid,CC(=O)NO,CC(=O)NO,0,Less-DILI-Concern,3,NaN
4,ACETYLCHOLINE CHLORIDE,acetylcholine chloride,acetylcholine chloride,CC(=O)OCC[N+](C)(C)C.[Cl-],CC(=O)OCC[N+](C)(C)C,0,No-DILI-concern,0,NaN
5,ACITRETIN,acitretin,acitretin,COc1cc(C)c(C=CC(C)=CC=CC(C)=CC(=O)O)c(C)c1C,CC1=CC(=C(C(=C1/C=C/C(=C/C=C/C(=C\C(=O)O)/C)/C...,1,Most-DILI-Concern,5,Yes
6,ADENOSINE,adenosine,adenosine,Nc1ncnc2c1ncn2C1OC(CO)C(O)C1O,C1=NC2=C(C(=N1)N)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,0,No-DILI-Concern,0,NaN
7,ALBENDAZOLE,albendazole,albendazole,CCCSc1ccc2nc(NC(=O)OC)[nH]c2c1,CCCSC1=CC2=C(C=C1)N=C(N2)NC(=O)OC,1,Most-DILI-Concern,7,Yes
8,ALLOPURINOL,allopurinol,allopurinol,O=c1[nH]cnc2[nH]ncc12,C1=C2C(=NC=NC2=O)NN1,1,Most-DILI-Concern,8,Yes
9,ALMOTRIPTAN,almotriptan,almotriptan,CN(C)CCc1c[nH]c2ccc(CS(=O)(=O)N3CCCC3)cc12,CN(C)CCC1=CNC2=C1C=C(C=C2)CS(=O)(=O)N3CCCC3,0,No-DILI-Concern,0,NaN


In [33]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import inchi

# =========================================================
# CANONICALIZE SMILES
# =========================================================

def smiles_to_inchikey(smiles):
    if pd.isna(smiles):
        return None

    try:
        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:
            return None

        return inchi.MolToInchiKey(mol)

    except Exception:
        return None


# Use the existing candidate_matches dataframe
identity_check = candidate_matches.copy()

identity_check["unitox_inchikey"] = (
    identity_check["smiles"].apply(smiles_to_inchikey)
)

identity_check["dilirank_inchikey"] = (
    identity_check["SMILES"].apply(smiles_to_inchikey)
)

# =========================================================
# CLASSIFY STRUCTURE MATCHES
# =========================================================

def compare_structures(row):
    key1 = row["unitox_inchikey"]
    key2 = row["dilirank_inchikey"]

    if pd.isna(key1) or pd.isna(key2):
        return "Structure unavailable/invalid"

    if key1 == key2:
        return "Exact InChIKey match"

    return "Name match, structure differs"


identity_check["structure_status"] = identity_check.apply(
    compare_structures,
    axis=1
)

print("=" * 70)
print("MOLECULAR IDENTITY AUDIT")
print("=" * 70)

print(identity_check["structure_status"].value_counts(dropna=False))

print("\nExamples where names match but structures differ:")

display(
    identity_check[
        identity_check["structure_status"]
        == "Name match, structure differs"
    ][
        [
            "generic_name",
            "LabelCompoundName",
            "unitox_inchikey",
            "dilirank_inchikey",
            "liver_toxicity_binary_rating_0_1",
            "DILIConcern",
            "AdjudicatedDILI"
        ]
    ].head(20)
)

print("\nExamples with exact structure matches:")

display(
    identity_check[
        identity_check["structure_status"] == "Exact InChIKey match"
    ][
        [
            "generic_name",
            "LabelCompoundName",
            "liver_toxicity_binary_rating_0_1",
            "DILIConcern",
            "AdjudicatedDILI"
        ]
    ].head(20)
)

MOLECULAR IDENTITY AUDIT
structure_status
Exact InChIKey match             253
Name match, structure differs    238
Structure unavailable/invalid      6
Name: count, dtype: int64

Examples where names match but structures differ:


,generic_name,LabelCompoundName,unitox_inchikey,dilirank_inchikey,liver_toxicity_binary_rating_0_1,DILIConcern,AdjudicatedDILI
0,ABACAVIR,abacavir,MCGSCOLBFJQGHM-UHFFFAOYSA-N,MCGSCOLBFJQGHM-PEHGTWAWSA-N,1,Most-DILI-Concern,Yes
1,ACARBOSE,acarbose,XUFXOAAUWZOOIT-UHFFFAOYSA-N,XUFXOAAUWZOOIT-JMPDRRIHSA-N,1,Most-DILI-Concern,Yes
4,ACETYLCHOLINE CHLORIDE,acetylcholine chloride,JUGOREOARAHOCO-UHFFFAOYSA-M,OIPILFWXSMYKGL-UHFFFAOYSA-N,0,No-DILI-concern,NaN
5,ACITRETIN,acitretin,IHUNBGSDBOWDMA-UHFFFAOYSA-N,IHUNBGSDBOWDMA-UGOGCBOOSA-N,1,Most-DILI-Concern,Yes
6,ADENOSINE,adenosine,OIRDTQYFTABQOQ-UHFFFAOYSA-N,OIRDTQYFTABQOQ-KQYNXXCUSA-N,0,No-DILI-Concern,NaN
16,AMOXICILLIN,amoxicillin,LSQZJLSUYDQPKJ-UHFFFAOYSA-N,LSQZJLSUYDQPKJ-NJBDSQKTSA-N,1,Less-DILI-Concern,Yes
17,AMPHOTERICIN B,amphotericin b,APKFDSVGJQXUKY-UHFFFAOYSA-N,APKFDSVGJQXUKY-KKGHZKTASA-N,0,Less-DILI-Concern,Yes
18,AMPICILLIN,ampicillin,AVKUERGKIZMTKX-UHFFFAOYSA-N,AVKUERGKIZMTKX-NJBDSQKTSA-N,0,Less-DILI-Concern,Yes
19,ANIDULAFUNGIN,anidulafungin,JHVAMHSQVVQIOT-UHFFFAOYSA-N,JHVAMHSQVVQIOT-MFAJLEFUSA-N,1,Most-DILI-Concern,NaN
20,APREPITANT,aprepitant,ATALOFNDEOCMKK-UHFFFAOYSA-N,ATALOFNDEOCMKK-BYYRLHKVSA-N,0,Less-DILI-Concern,NaN



Examples with exact structure matches:


,generic_name,LabelCompoundName,liver_toxicity_binary_rating_0_1,DILIConcern,AdjudicatedDILI
2,ACETAZOLAMIDE,acetazolamide,1,Most-DILI-Concern,Yes
3,ACETOHYDROXAMIC ACID,acetohydroxamic acid,0,Less-DILI-Concern,NaN
7,ALBENDAZOLE,albendazole,1,Most-DILI-Concern,Yes
8,ALLOPURINOL,allopurinol,1,Most-DILI-Concern,Yes
9,ALMOTRIPTAN,almotriptan,0,No-DILI-Concern,NaN
10,AMANTADINE,amantadine,0,Less-DILI-Concern,NaN
11,AMBRISENTAN,ambrisentan,0,Less-DILI-Concern,NaN
12,AMIFOSTINE,amifostine,0,No-DILI-Concern,NaN
13,AMINOCAPROIC ACID,aminocaproic acid,1,No-DILI-Concern,NaN
14,AMINOPHYLLINE,aminophylline,1,No-DILI-Concern,NaN


In [34]:
print("=" * 60)
print("MOLECULAR IDENTITY AUDIT")
print("=" * 60)

print(identity_check["structure_status"].value_counts(dropna=False))

print("\nTotal candidate rows:", len(identity_check))
print("Unique matched drug names:",
      identity_check["normalized_name"].nunique())

print("\nMissing UniTox InChIKeys:",
      identity_check["unitox_inchikey"].isna().sum())

print("Missing DILIrank InChIKeys:",
      identity_check["dilirank_inchikey"].isna().sum())

MOLECULAR IDENTITY AUDIT
structure_status
Exact InChIKey match             253
Name match, structure differs    238
Structure unavailable/invalid      6
Name: count, dtype: int64

Total candidate rows: 497
Unique matched drug names: 497

Missing UniTox InChIKeys: 0
Missing DILIrank InChIKeys: 6


In [35]:
print("DILI concern categories:")
print(dilirank["DILIConcern"].value_counts(dropna=False))

print("\nAdjudication categories:")
print(dilirank["AdjudicatedDILI"].value_counts(dropna=False))

print("\nCross-tabulation:")
display(
    pd.crosstab(
        dilirank["DILIConcern"],
        dilirank["AdjudicatedDILI"],
        margins=True,
        dropna=False
    )
)

DILI concern categories:
DILIConcern
Less-DILI-Concern    488
No-DILI-Concern      321
Most-DILI-Concern    208
No-DILI-concern       18
Most-DILI-concern      1
Name: count, dtype: int64

Adjudication categories:
AdjudicatedDILI
NaN                625
Yes                399
Yes(literature)     12
Name: count, dtype: int64

Cross-tabulation:


AdjudicatedDILI,Yes,Yes(literature),NaN,All
DILIConcern,,,,
Less-DILI-Concern,249,2,237,488
Most-DILI-Concern,124,9,75,208
Most-DILI-concern,0,0,1,1
No-DILI-Concern,26,1,294,321
No-DILI-concern,0,0,18,18
All,399,12,625,1036


In [36]:
# Inspect name matches with different InChIKeys

mismatch_cols = [
    "generic_name",
    "LabelCompoundName",
    "smiles",
    "SMILES",
    "unitox_inchikey",
    "dilirank_inchikey",
    "liver_toxicity_binary_rating_0_1",
    "DILIConcern",
    "AdjudicatedDILI"
]

mismatches = identity_check[
    identity_check["structure_status"] == "Name match, structure differs"
].copy()

print("Total mismatches:", len(mismatches))

display(mismatches[mismatch_cols].head(30))

Total mismatches: 238


,generic_name,LabelCompoundName,smiles,SMILES,unitox_inchikey,dilirank_inchikey,liver_toxicity_binary_rating_0_1,DILIConcern,AdjudicatedDILI
0,ABACAVIR,abacavir,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,C1CC1NC2=NC(=NC3=C2N=CN3C4C[C@H](C=C4)CO)N,MCGSCOLBFJQGHM-UHFFFAOYSA-N,MCGSCOLBFJQGHM-PEHGTWAWSA-N,1,Most-DILI-Concern,Yes
1,ACARBOSE,acarbose,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,C[C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O[C@@H]2[C...,XUFXOAAUWZOOIT-UHFFFAOYSA-N,XUFXOAAUWZOOIT-JMPDRRIHSA-N,1,Most-DILI-Concern,Yes
4,ACETYLCHOLINE CHLORIDE,acetylcholine chloride,CC(=O)OCC[N+](C)(C)C.[Cl-],CC(=O)OCC[N+](C)(C)C,JUGOREOARAHOCO-UHFFFAOYSA-M,OIPILFWXSMYKGL-UHFFFAOYSA-N,0,No-DILI-concern,NaN
5,ACITRETIN,acitretin,COc1cc(C)c(C=CC(C)=CC=CC(C)=CC(=O)O)c(C)c1C,CC1=CC(=C(C(=C1/C=C/C(=C/C=C/C(=C\C(=O)O)/C)/C...,IHUNBGSDBOWDMA-UHFFFAOYSA-N,IHUNBGSDBOWDMA-UGOGCBOOSA-N,1,Most-DILI-Concern,Yes
6,ADENOSINE,adenosine,Nc1ncnc2c1ncn2C1OC(CO)C(O)C1O,C1=NC2=C(C(=N1)N)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,OIRDTQYFTABQOQ-UHFFFAOYSA-N,OIRDTQYFTABQOQ-KQYNXXCUSA-N,0,No-DILI-Concern,NaN
16,AMOXICILLIN,amoxicillin,CC1(C)SC2C(NC(=O)C(N)c3ccc(O)cc3)C(=O)N2C1C(=O)O,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,LSQZJLSUYDQPKJ-UHFFFAOYSA-N,LSQZJLSUYDQPKJ-NJBDSQKTSA-N,1,Less-DILI-Concern,Yes
17,AMPHOTERICIN B,amphotericin b,CC1C=CC=CC=CC=CC=CC=CC=CC(OC2OC(C)C(O)C(N)C2O)...,C[C@H]1C=CC=CC=CC=CC=CC=CC=C[C@@H](C[C@H]2[C@@...,APKFDSVGJQXUKY-UHFFFAOYSA-N,APKFDSVGJQXUKY-KKGHZKTASA-N,0,Less-DILI-Concern,Yes
18,AMPICILLIN,ampicillin,CC1(C)SC2C(NC(=O)C(N)c3ccccc3)C(=O)N2C1C(=O)O,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,AVKUERGKIZMTKX-UHFFFAOYSA-N,AVKUERGKIZMTKX-NJBDSQKTSA-N,0,Less-DILI-Concern,Yes
19,ANIDULAFUNGIN,anidulafungin,CCCCCOc1ccc(-c2ccc(-c3ccc(C(=O)NC4CC(O)C(O)NC(...,CCCCCOC1=CC=C(C=C1)C2=CC=C(C=C2)C3=CC=C(C=C3)C...,JHVAMHSQVVQIOT-UHFFFAOYSA-N,JHVAMHSQVVQIOT-MFAJLEFUSA-N,1,Most-DILI-Concern,NaN
20,APREPITANT,aprepitant,CC(OC1OCCN(Cc2n[nH]c(=O)[nH]2)C1c1ccc(F)cc1)c1...,C[C@H](C1=CC(=CC(=C1)C(F)(F)F)C(F)(F)F)O[C@H]2...,ATALOFNDEOCMKK-UHFFFAOYSA-N,ATALOFNDEOCMKK-BYYRLHKVSA-N,0,Less-DILI-Concern,NaN


In [37]:
from rdkit import Chem

def structure_summary(smiles):
    if pd.isna(smiles):
        return None

    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    return {
        "canonical_smiles": Chem.MolToSmiles(mol, canonical=True),
        "components": len(Chem.GetMolFrags(mol)),
        "heavy_atoms": mol.GetNumHeavyAtoms()
    }

# Apply to the first 30 mismatches for inspection
sample = mismatches.head(30).copy()

sample["unitox_structure"] = sample["smiles"].apply(structure_summary)
sample["dilirank_structure"] = sample["SMILES"].apply(structure_summary)

display(
    sample[
        [
            "generic_name",
            "unitox_structure",
            "dilirank_structure"
        ]
    ]
)

,generic_name,unitox_structure,dilirank_structure
0,ABACAVIR,{'canonical_smiles': 'Nc1nc(NC2CC2)c2ncn(C3C=C...,{'canonical_smiles': 'Nc1nc(NC2CC2)c2ncn(C3C=C...
1,ACARBOSE,{'canonical_smiles': 'CC1OC(OC2C(CO)OC(OC3C(CO...,{'canonical_smiles': 'C[C@H]1O[C@H](O[C@H]2[C@...
4,ACETYLCHOLINE CHLORIDE,{'canonical_smiles': 'CC(=O)OCC[N+](C)(C)C.[Cl...,"{'canonical_smiles': 'CC(=O)OCC[N+](C)(C)C', '..."
5,ACITRETIN,{'canonical_smiles': 'COc1cc(C)c(C=CC(C)=CC=CC...,{'canonical_smiles': 'COc1cc(C)c(/C=C/C(C)=C/C...
6,ADENOSINE,{'canonical_smiles': 'Nc1ncnc2c1ncn2C1OC(CO)C(...,{'canonical_smiles': 'Nc1ncnc2c1ncn2[C@@H]1O[C...
16,AMOXICILLIN,{'canonical_smiles': 'CC1(C)SC2C(NC(=O)C(N)c3c...,{'canonical_smiles': 'CC1(C)S[C@@H]2[C@H](NC(=...
17,AMPHOTERICIN B,{'canonical_smiles': 'CC1C=CC=CC=CC=CC=CC=CC=C...,{'canonical_smiles': 'C[C@@H]1[C@H](O)[C@@H](C...
18,AMPICILLIN,{'canonical_smiles': 'CC1(C)SC2C(NC(=O)C(N)c3c...,{'canonical_smiles': 'CC1(C)S[C@@H]2[C@H](NC(=...
19,ANIDULAFUNGIN,{'canonical_smiles': 'CCCCCOc1ccc(-c2ccc(-c3cc...,{'canonical_smiles': 'CCCCCOc1ccc(-c2ccc(-c3cc...
20,APREPITANT,{'canonical_smiles': 'CC(OC1OCCN(Cc2n[nH]c(=O)...,{'canonical_smiles': 'C[C@@H](O[C@@H]1OCCN(Cc2...


In [38]:

# Classify chemical identity matches more carefully

def classify_inchikeys(row):
    k1 = row["unitox_inchikey"]
    k2 = row["dilirank_inchikey"]

    if pd.isna(k1) or pd.isna(k2):
        return "Missing InChIKey"

    if k1 == k2:
        return "Exact InChIKey match"

    # First 14 characters encode the connectivity block
    if k1[:14] == k2[:14]:
        return "Same connectivity block; stereochemistry/other layers differ"

    return "Different connectivity block"

identity_check["identity_class"] = identity_check.apply(
    classify_inchikeys, axis=1
)

print(identity_check["identity_class"].value_counts())

# Inspect the first 20 cases with matching connectivity
connectivity_cases = identity_check[
    identity_check["identity_class"]
    == "Same connectivity block; stereochemistry/other layers differ"
]

display(
    connectivity_cases[
        [
            "generic_name",
            "unitox_inchikey",
            "dilirank_inchikey",
            "liver_toxicity_binary_rating_0_1",
            "DILIConcern",
            "AdjudicatedDILI",
        ]
    ].head(20)
)


identity_class
Exact InChIKey match                                            253
Same connectivity block; stereochemistry/other layers differ    193
Different connectivity block                                     45
Missing InChIKey                                                  6
Name: count, dtype: int64


,generic_name,unitox_inchikey,dilirank_inchikey,liver_toxicity_binary_rating_0_1,DILIConcern,AdjudicatedDILI
0,ABACAVIR,MCGSCOLBFJQGHM-UHFFFAOYSA-N,MCGSCOLBFJQGHM-PEHGTWAWSA-N,1,Most-DILI-Concern,Yes
1,ACARBOSE,XUFXOAAUWZOOIT-UHFFFAOYSA-N,XUFXOAAUWZOOIT-JMPDRRIHSA-N,1,Most-DILI-Concern,Yes
5,ACITRETIN,IHUNBGSDBOWDMA-UHFFFAOYSA-N,IHUNBGSDBOWDMA-UGOGCBOOSA-N,1,Most-DILI-Concern,Yes
6,ADENOSINE,OIRDTQYFTABQOQ-UHFFFAOYSA-N,OIRDTQYFTABQOQ-KQYNXXCUSA-N,0,No-DILI-Concern,NaN
16,AMOXICILLIN,LSQZJLSUYDQPKJ-UHFFFAOYSA-N,LSQZJLSUYDQPKJ-NJBDSQKTSA-N,1,Less-DILI-Concern,Yes
17,AMPHOTERICIN B,APKFDSVGJQXUKY-UHFFFAOYSA-N,APKFDSVGJQXUKY-KKGHZKTASA-N,0,Less-DILI-Concern,Yes
18,AMPICILLIN,AVKUERGKIZMTKX-UHFFFAOYSA-N,AVKUERGKIZMTKX-NJBDSQKTSA-N,0,Less-DILI-Concern,Yes
19,ANIDULAFUNGIN,JHVAMHSQVVQIOT-UHFFFAOYSA-N,JHVAMHSQVVQIOT-MFAJLEFUSA-N,1,Most-DILI-Concern,NaN
20,APREPITANT,ATALOFNDEOCMKK-UHFFFAOYSA-N,ATALOFNDEOCMKK-BYYRLHKVSA-N,0,Less-DILI-Concern,NaN
21,ARGATROBAN,KXNPVXPOPUZYGB-UHFFFAOYSA-N,KXNPVXPOPUZYGB-XYVMCAHJSA-N,1,No-DILI-Concern,NaN


In [39]:

# 1. Count the chemical identity groups
print("Chemical identity classification:")
print(identity_check["identity_class"].value_counts(dropna=False))

# 2. Normalize DILIrank category capitalization and whitespace
identity_check["DILIConcern_clean"] = (
    identity_check["DILIConcern"]
    .astype("string")
    .str.strip()
    .str.lower()
)

identity_check["DILIConcern_clean"] = (
    identity_check["DILIConcern_clean"]
    .replace({
        "most-dili-concern": "Most",
        "less-dili-concern": "Less",
        "no-dili-concern": "No"
    })
)

# 3. First, use only exact full InChIKey matches
exact = identity_check[
    identity_check["identity_class"] == "Exact InChIKey match"
].copy()

print("\nExact-match sample size:", len(exact))
print("\nUniTox label vs DILIrank category:")
display(pd.crosstab(
    exact["DILIConcern_clean"],
    exact["liver_toxicity_binary_rating_0_1"],
    margins=True
))


Chemical identity classification:
identity_class
Exact InChIKey match                                            253
Same connectivity block; stereochemistry/other layers differ    193
Different connectivity block                                     45
Missing InChIKey                                                  6
Name: count, dtype: int64

Exact-match sample size: 253

UniTox label vs DILIrank category:


liver_toxicity_binary_rating_0_1,0,1,All
DILIConcern_clean,,,
Less,37,100,137
Most,2,53,55
No,47,14,61
All,86,167,253


In [40]:

from scipy.stats import spearmanr, chi2_contingency
import numpy as np
import pandas as pd

# Use the exact chemical identity matches only
analysis_df = exact.copy()

# Explicit ordinal encoding; Less remains a separate category
analysis_df["dili_ordinal"] = (
    analysis_df["DILIConcern_clean"]
    .map({"No": 0, "Less": 1, "Most": 2})
)

analysis_df["unitox_label"] = pd.to_numeric(
    analysis_df["liver_toxicity_binary_rating_0_1"],
    errors="coerce"
)

analysis_df = analysis_df.dropna(
    subset=["dili_ordinal", "unitox_label"]
)

# Spearman rank correlation
rho, spearman_p = spearmanr(
    analysis_df["dili_ordinal"],
    analysis_df["unitox_label"]
)

# Chi-square test for the 3 x 2 table
table = pd.crosstab(
    analysis_df["DILIConcern_clean"],
    analysis_df["unitox_label"]
).reindex(
    index=["No", "Less", "Most"],
    columns=[0, 1],
    fill_value=0
)

chi2, chi_p, dof, expected = chi2_contingency(table)

print("N =", len(analysis_df))
print("\nContingency table:")
display(table)

print(f"Spearman rho = {rho:.4f}")
print(f"Spearman p-value = {spearman_p:.6g}")
print(f"Chi-square statistic = {chi2:.4f}")
print(f"Chi-square degrees of freedom = {dof}")
print(f"Chi-square p-value = {chi_p:.6g}")
print("\nExpected cell counts:")
display(pd.DataFrame(
    expected,
    index=table.index,
    columns=table.columns
))


N = 253

Contingency table:


unitox_label,0,1
DILIConcern_clean,,
No,47,14
Less,37,100
Most,2,53


Spearman rho = 0.5313
Spearman p-value = 7.86018e-20
Chi-square statistic = 75.9682
Chi-square degrees of freedom = 2
Chi-square p-value = 3.18945e-17

Expected cell counts:


unitox_label,0,1
DILIConcern_clean,,
No,20.735178,40.264822
Less,46.569170,90.430830
Most,18.695652,36.304348


In [41]:

from scipy.stats import chi2_contingency, spearmanr
import pandas as pd

# Use the existing strict exact-match subset
analysis_df = exact.copy()

analysis_df["dili_ordinal"] = (
    analysis_df["DILIConcern_clean"]
    .map({"No": 0, "Less": 1, "Most": 2})
)

analysis_df["unitox_label"] = pd.to_numeric(
    analysis_df["liver_toxicity_binary_rating_0_1"],
    errors="coerce"
)

analysis_df = analysis_df.dropna(
    subset=["dili_ordinal", "unitox_label"]
)

# Observed counts
observed = pd.crosstab(
    analysis_df["DILIConcern_clean"],
    analysis_df["unitox_label"]
).reindex(
    index=["No", "Less", "Most"],
    columns=[0, 1],
    fill_value=0
)

# Chi-square test
chi2, p_chi, dof, expected = chi2_contingency(observed)

# Spearman correlation
rho, p_spearman = spearmanr(
    analysis_df["dili_ordinal"],
    analysis_df["unitox_label"]
)

print("N =", len(analysis_df))
print("\nObserved counts:")
display(observed)

print("\nExpected counts under independence:")
display(pd.DataFrame(
    expected,
    index=observed.index,
    columns=observed.columns
))

print(f"\nChi-square = {chi2:.4f}")
print(f"Degrees of freedom = {dof}")
print(f"Chi-square p-value = {p_chi:.6g}")
print(f"Spearman rho = {rho:.4f}")
print(f"Spearman p-value = {p_spearman:.6g}")

# Clear-extremes sensitivity analysis: exclude Less
extremes = analysis_df[
    analysis_df["DILIConcern_clean"].isin(["No", "Most"])
].copy()

extreme_table = pd.crosstab(
    extremes["DILIConcern_clean"],
    extremes["unitox_label"]
).reindex(
    index=["No", "Most"],
    columns=[0, 1],
    fill_value=0
)

print("\nClear-extremes table (Less excluded):")
display(extreme_table)

print("\nPositive-label rate by category:")
display(
    extremes.groupby("DILIConcern_clean")["unitox_label"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "positive_rate"})
)


N = 253

Observed counts:


unitox_label,0,1
DILIConcern_clean,,
No,47,14
Less,37,100
Most,2,53



Expected counts under independence:


unitox_label,0,1
DILIConcern_clean,,
No,20.735178,40.264822
Less,46.569170,90.430830
Most,18.695652,36.304348



Chi-square = 75.9682
Degrees of freedom = 2
Chi-square p-value = 3.18945e-17
Spearman rho = 0.5313
Spearman p-value = 7.86018e-20

Clear-extremes table (Less excluded):


unitox_label,0,1
DILIConcern_clean,,
No,47,14
Most,2,53



Positive-label rate by category:


,count,positive_rate
DILIConcern_clean,,
Most,55,0.963636
No,61,0.229508


In [42]:

from scipy.stats import fisher_exact
import numpy as np

# DILIrank Most = reference positive; No = reference negative
# UniTox 1 = predicted positive; UniTox 0 = predicted negative
tn = 47  # DILIrank No, UniTox 0
fp = 14  # DILIrank No, UniTox 1
fn = 2   # DILIrank Most, UniTox 0
tp = 53  # DILIrank Most, UniTox 1

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision = tp / (tp + fp)
npv = tn / (tn + fn)
accuracy = (tp + tn) / (tp + tn + fp + fn)

odds_ratio, fisher_p = fisher_exact(
    [[tp, fn], [fp, tn]],
    alternative="two-sided"
)

print(f"Sample size: {tp + tn + fp + fn}")
print(f"Raw agreement: {accuracy:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Positive predictive value: {precision:.4f}")
print(f"Negative predictive value: {npv:.4f}")
print(f"Odds ratio: {odds_ratio:.4f}")
print(f"Fisher exact p-value: {fisher_p:.6g}")


Sample size: 116
Raw agreement: 0.8621
Sensitivity: 0.9636
Specificity: 0.7705
Positive predictive value: 0.7910
Negative predictive value: 0.9592
Odds ratio: 88.9643
Fisher exact p-value: 3.2226e-17


In [43]:

import numpy as np
import pandas as pd

# Define the target and molecular structure columns
TARGET = "liver_toxicity_binary_rating_0_1"
SMILES_COL = "smiles"

# Make a separate working copy; preserve the original dataframe
work_df = df.copy().reset_index(drop=True)

print("=== DATASET AUDIT ===")
print("Dataset shape:", work_df.shape)
print("Target column exists:", TARGET in work_df.columns)
print("SMILES column exists:", SMILES_COL in work_df.columns)

if TARGET not in work_df.columns or SMILES_COL not in work_df.columns:
    raise ValueError("Required target or SMILES column is missing.")

# Convert target to numeric for validation
work_df[TARGET] = pd.to_numeric(work_df[TARGET], errors="coerce")

print("\n=== TARGET DISTRIBUTION ===")
print(work_df[TARGET].value_counts(dropna=False).sort_index())

print("\n=== MISSING VALUES ===")
print("Missing targets:", work_df[TARGET].isna().sum())
print("Missing SMILES:", work_df[SMILES_COL].isna().sum())
print("Duplicate SMILES strings:", work_df[SMILES_COL].duplicated().sum())

# Confirm that labels are binary and non-missing
valid_labels = set(work_df[TARGET].dropna().unique())
print("\nObserved non-missing label values:", valid_labels)

if work_df[TARGET].isna().any():
    print("WARNING: Missing target labels need investigation.")

if not valid_labels.issubset({0, 1}):
    print("WARNING: Target contains values other than 0 and 1.")

# Summarize the external-reference subset
print("\n=== EXTERNAL REFERENCE ===")
print("Exact InChIKey matches:", len(exact))
print("DILIrank category counts:")
print(exact["DILIConcern_clean"].value_counts(dropna=False))

print("\nAudit completed.")


=== DATASET AUDIT ===
Dataset shape: (1349, 44)
Target column exists: True
SMILES column exists: True

=== TARGET DISTRIBUTION ===
liver_toxicity_binary_rating_0_1
0    613
1    736
Name: count, dtype: int64

=== MISSING VALUES ===
Missing targets: 0
Missing SMILES: 0
Duplicate SMILES strings: 0

Observed non-missing label values: {np.int64(0), np.int64(1)}

=== EXTERNAL REFERENCE ===
Exact InChIKey matches: 253
DILIrank category counts:
DILIConcern_clean
Less    137
No       61
Most     55
Name: count, dtype: Int64

Audit completed.


In [44]:

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

# Create a reproducible working dataset
work_df = work_df.reset_index(drop=True).copy()

# Generate a Bemis-Murcko scaffold for each molecule
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold, canonical=True)

work_df["scaffold"] = work_df[SMILES_COL].apply(get_scaffold)

if work_df["scaffold"].isna().any():
    raise ValueError("Invalid structures found. Investigate before splitting.")

print("Total drugs:", len(work_df))
print("Unique Murcko scaffolds:", work_df["scaffold"].nunique())
print("Largest scaffold groups:")
print(work_df["scaffold"].value_counts().head(10))

# Empty scaffold is retained as a group:
# many acyclic compounds have no Bemis-Murcko scaffold.
print("\nDrugs with empty scaffold:",
      int((work_df["scaffold"] == "").sum()))

# Assign whole scaffold groups to 10 folds.
# Greedy assignment balances fold sizes, while preserving scaffold groups.
def assign_scaffold_folds(data, n_splits=10, random_state=42):
    rng = np.random.default_rng(random_state)

    y = data[TARGET].to_numpy(dtype=int)
    scaffold_values = data["scaffold"].to_numpy()

    groups = {}
    for idx, scaffold in enumerate(scaffold_values):
        groups.setdefault(scaffold, []).append(idx)

    scaffold_items = list(groups.items())
    rng.shuffle(scaffold_items)

    # Large groups first; randomization resolves equal-size ties
    scaffold_items.sort(key=lambda item: len(item[1]), reverse=True)

    fold_indices = [[] for _ in range(n_splits)]

    # Keep approximate class counts and sizes balanced
    fold_class_counts = np.zeros((n_splits, 2), dtype=int)

    for scaffold, indices in scaffold_items:
        labels = y[indices]
        group_counts = np.bincount(labels, minlength=2)

        scores = []
        for fold in range(n_splits):
            proposed = fold_class_counts.copy()
            proposed[fold] += group_counts

            # Penalize imbalance in total class counts across folds
            class_imbalance = np.sum(
                np.std(proposed, axis=0)
            )

            size_imbalance = np.std(
                [len(fold_indices[j]) +
                 (len(indices) if j == fold else 0)
                 for j in range(n_splits)]
            )

            scores.append(class_imbalance + 0.01 * size_imbalance)

        best_fold = int(np.argmin(scores))
        fold_indices[best_fold].extend(indices)
        fold_class_counts[best_fold] += group_counts

    fold_id = np.full(len(data), -1, dtype=int)
    for fold, indices in enumerate(fold_indices):
        fold_id[indices] = fold

    return fold_id

work_df["fold"] = assign_scaffold_folds(
    work_df, n_splits=10, random_state=42
)

# Verify fold sizes and class counts
print("\n=== FOLD SUMMARY ===")
fold_summary = []

for fold in range(10):
    subset = work_df[work_df["fold"] == fold]
    fold_summary.append({
        "fold": fold,
        "n_drugs": len(subset),
        "negative": int((subset[TARGET] == 0).sum()),
        "positive": int((subset[TARGET] == 1).sum()),
        "unique_scaffolds": subset["scaffold"].nunique()
    })

display(pd.DataFrame(fold_summary))

# Verify no scaffold is shared between folds
scaffold_fold_counts = work_df.groupby("scaffold")["fold"].nunique()
leaked_scaffolds = int((scaffold_fold_counts > 1).sum())

print("\nScaffolds appearing in multiple folds:", leaked_scaffolds)

assert work_df["fold"].ge(0).all()
assert leaked_scaffolds == 0

print("Scaffold leakage check passed.")


[19:32:26] WARNING: not removing hydrogen atom without neighbors
[19:32:26] WARNING: not removing hydrogen atom without neighbors
[19:32:26] WARNING: not removing hydrogen atom without neighbors
[19:32:26] WARNING: not removing hydrogen atom without neighbors
[19:32:26] WARNING: not removing hydrogen atom without neighbors
[19:32:26] WARNING: not removing hydrogen atom without neighbors
[19:32:26] WARNING: not removing hydrogen atom without neighbors
[19:32:26] WARNING: not removing hydrogen atom without neighbors
[19:32:26] WARNING: not removing hydrogen atom without neighbors
[19:32:26] WARNING: not removing hydrogen atom without neighbors
[19:32:29] WARNING: not removing hydrogen atom without neighbors
[19:32:29] WARNING: not removing hydrogen atom without neighbors


Total drugs: 1349
Unique Murcko scaffolds: 802
Largest scaffold groups:
scaffold
                                      122
c1ccccc1                               77
O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21        15
O=C1C=C2CCC3C4CCCC4CCC3C2CC1           12
O=C1C=CC2C(=O)C3=Cc4ccccc4CC3CC2C1     10
c1ccncc1                                8
c1ccccc1.c1ccccc1                       8
c1ccc(Cc2ccccc2)cc1                     8
c1ccc2c(c1)Sc1ccccc1N2CCCN1CCNCC1       8
c1c[nH]cn1                              7
Name: count, dtype: int64

Drugs with empty scaffold: 122

=== FOLD SUMMARY ===


,fold,n_drugs,negative,positive,unique_scaffolds
0,0,163,90,73,40
1,1,132,58,74,56
2,2,132,58,74,87
3,3,132,58,74,88
4,4,132,58,74,89
5,5,132,58,74,88
6,6,132,59,73,89
7,7,132,58,74,89
8,8,131,58,73,88
9,9,131,58,73,88



Scaffolds appearing in multiple folds: 0
Scaffold leakage check passed.


In [45]:

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

def validate_molecule(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return False
    return True

work_df["smiles_valid"] = work_df[SMILES_COL].apply(validate_molecule)

print("Invalid SMILES:", int((~work_df["smiles_valid"]).sum()))
print("Valid SMILES:", int(work_df["smiles_valid"].sum()))

assert work_df["smiles_valid"].all(), (
    "Investigate invalid structures before proceeding."
)


[19:44:34] WARNING: not removing hydrogen atom without neighbors
[19:44:34] WARNING: not removing hydrogen atom without neighbors
[19:44:34] WARNING: not removing hydrogen atom without neighbors
[19:44:34] WARNING: not removing hydrogen atom without neighbors
[19:44:34] WARNING: not removing hydrogen atom without neighbors
[19:44:34] WARNING: not removing hydrogen atom without neighbors
[19:44:34] WARNING: not removing hydrogen atom without neighbors
[19:44:34] WARNING: not removing hydrogen atom without neighbors
[19:44:34] WARNING: not removing hydrogen atom without neighbors
[19:44:34] WARNING: not removing hydrogen atom without neighbors


Invalid SMILES: 0
Valid SMILES: 1349


[19:44:35] WARNING: not removing hydrogen atom without neighbors
[19:44:35] WARNING: not removing hydrogen atom without neighbors


In [46]:

# Save the exact split assignment for reproducibility
fold_file = "unitox_liver_scaffold_folds.csv"

work_df[
    ["generic_name", SMILES_COL, TARGET, "scaffold", "fold"]
].to_csv(fold_file, index=False)

print("Saved fold assignments to:", fold_file)


Saved fold assignments to: unitox_liver_scaffold_folds.csv


In [47]:

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# Ensure the working dataframe has a clean positional index
work_df = work_df.reset_index(drop=True).copy()

# Generate Morgan fingerprints (radius 2, 2048 bits)
fp_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    fp = fp_generator.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

X = np.vstack(work_df[SMILES_COL].apply(smiles_to_fp).to_numpy())
y = work_df[TARGET].to_numpy(dtype=int)
folds = work_df["fold"].to_numpy(dtype=int)

print("Fingerprint matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Fold IDs:", np.unique(folds))

# Store exactly one out-of-fold probability for every drug
oof_probability = np.full(len(work_df), np.nan)
fold_results = []

for fold in sorted(np.unique(folds)):
    train_idx = np.where(folds != fold)[0]
    test_idx = np.where(folds == fold)[0]

    model = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    )

    model.fit(X[train_idx], y[train_idx])

    # Probability of the positive class
    positive_col = list(model.classes_).index(1)
    probabilities = model.predict_proba(X[test_idx])[:, positive_col]

    oof_probability[test_idx] = probabilities

    # AUC is undefined if a held-out fold has only one class
    if len(np.unique(y[test_idx])) == 2:
        fold_auc = roc_auc_score(y[test_idx], probabilities)
    else:
        fold_auc = np.nan

    fold_results.append({
        "fold": int(fold),
        "train_n": len(train_idx),
        "test_n": len(test_idx),
        "test_auc": fold_auc
    })

assert np.isfinite(oof_probability).all(), (
    "Some drugs do not have an out-of-fold prediction."
)

# Basic performance and agreement summaries
oof_prediction = (oof_probability >= 0.5).astype(int)

print("\n=== OUT-OF-FOLD RESULTS ===")
print("Number of predictions:", len(oof_probability))
print("Pooled OOF ROC-AUC:", round(
    roc_auc_score(y, oof_probability), 4
))
print("Pooled OOF accuracy:", round(
    accuracy_score(y, oof_prediction), 4
))

print("\nFold-level results:")
display(pd.DataFrame(fold_results))

# Save predictions for the next reliability-score stage
oof_df = work_df[
    ["generic_name", SMILES_COL, TARGET, "scaffold", "fold"]
].copy()

oof_df["oof_probability"] = oof_probability
oof_df["oof_prediction"] = oof_prediction
oof_df["oof_agrees_with_label"] = (
    oof_df["oof_prediction"] == oof_df[TARGET]
)

# Confidence in the model's binary decision, from 0 to 1
oof_df["oof_model_confidence"] = np.abs(
    2 * oof_df["oof_probability"] - 1
)

oof_df.to_csv("unitox_liver_oof_predictions.csv", index=False)

print("\nSaved: unitox_liver_oof_predictions.csv")
print("\nModel agreement with existing labels:")
display(
    oof_df["oof_agrees_with_label"]
    .value_counts()
    .rename(index={True: "Agrees", False: "Disagrees"})
    .to_frame("number_of_drugs")
)


[19:53:59] WARNING: not removing hydrogen atom without neighbors
[19:53:59] WARNING: not removing hydrogen atom without neighbors
[19:53:59] WARNING: not removing hydrogen atom without neighbors
[19:53:59] WARNING: not removing hydrogen atom without neighbors
[19:53:59] WARNING: not removing hydrogen atom without neighbors
[19:53:59] WARNING: not removing hydrogen atom without neighbors
[19:53:59] WARNING: not removing hydrogen atom without neighbors
[19:53:59] WARNING: not removing hydrogen atom without neighbors
[19:53:59] WARNING: not removing hydrogen atom without neighbors
[19:53:59] WARNING: not removing hydrogen atom without neighbors
[19:54:00] WARNING: not removing hydrogen atom without neighbors
[19:54:00] WARNING: not removing hydrogen atom without neighbors


Fingerprint matrix shape: (1349, 2048)
Target shape: (1349,)
Fold IDs: [0 1 2 3 4 5 6 7 8 9]

=== OUT-OF-FOLD RESULTS ===
Number of predictions: 1349
Pooled OOF ROC-AUC: 0.7204
Pooled OOF accuracy: 0.6723

Fold-level results:


,fold,train_n,test_n,test_auc
0,0,1186,163,0.716134
1,1,1217,132,0.709576
2,2,1217,132,0.738234
3,3,1217,132,0.721808
4,4,1217,132,0.732992
5,5,1217,132,0.592032
6,6,1217,132,0.680984
7,7,1217,132,0.803355
8,8,1218,131,0.751771
9,9,1218,131,0.681153



Saved: unitox_liver_oof_predictions.csv

Model agreement with existing labels:


,number_of_drugs
oof_agrees_with_label,
Agrees,907
Disagrees,442


In [48]:

# Overall performance
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd
import numpy as np

print("Pooled OOF ROC-AUC:",
      round(roc_auc_score(y, oof_probability), 4))
print("Pooled OOF accuracy:",
      round(accuracy_score(y, oof_prediction), 4))

# Confidence distribution by agreement status
print("\nConfidence by agreement with UniTox label:")
display(
    oof_df.groupby("oof_agrees_with_label")[
        "oof_model_confidence"
    ].agg(["count", "mean", "median", "std"])
)

# Confidence distribution by original UniTox label
print("\nConfidence by UniTox class:")
display(
    oof_df.groupby(TARGET)["oof_model_confidence"]
    .agg(["count", "mean", "median", "std"])
)

# How often does the model disagree within each class?
print("\nAgreement rate by UniTox class:")
display(
    oof_df.groupby(TARGET)["oof_agrees_with_label"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "agreement_rate"})
)


Pooled OOF ROC-AUC: 0.7204
Pooled OOF accuracy: 0.6723

Confidence by agreement with UniTox label:


,count,mean,median,std
oof_agrees_with_label,,,,
False,442,0.265103,0.236,0.179759
True,907,0.346853,0.328,0.202884



Confidence by UniTox class:


,count,mean,median,std
liver_toxicity_binary_rating_0_1,,,,
0,613,0.295996,0.256,0.199477
1,736,0.340117,0.328,0.197004



Agreement rate by UniTox class:


,count,agreement_rate
liver_toxicity_binary_rating_0_1,,
0,613,0.505710
1,736,0.811141


In [50]:

# Diagnose duplicate chemical identifiers
print("UniTox rows:", len(oof_df))
print("Unique non-missing InChIKeys:", oof_df["inchikey"].nunique())
print("Rows with duplicate InChIKeys:",
      int(oof_df["inchikey"].duplicated(keep=False).sum()))

duplicate_keys = (
    oof_df[oof_df["inchikey"].duplicated(keep=False)]
    .sort_values("inchikey")
)

display(
    duplicate_keys[
        ["generic_name", "inchikey", TARGET, "fold"]
    ].head(30)
)

# Check whether duplicated keys have conflicting toxicity labels
label_conflicts = (
    duplicate_keys.groupby("inchikey")[TARGET]
    .nunique()
)

print(
    "Duplicated InChIKeys with conflicting UniTox labels:",
    int((label_conflicts > 1).sum())
)


UniTox rows: 1349
Unique non-missing InChIKeys: 1348
Rows with duplicate InChIKeys: 2


,generic_name,inchikey,liver_toxicity_binary_rating_0_1,fold
986,ATAZANAVIR SULFATE,DQSGVVGOPRWTKI-UHFFFAOYSA-N,1,7
1276,REYATAZ,DQSGVVGOPRWTKI-UHFFFAOYSA-N,1,7


Duplicated InChIKeys with conflicting UniTox labels: 0


In [51]:

# Build one external record per full InChIKey
external = exact[
    ["unitox_inchikey", "DILIConcern_clean"]
].copy()

external = external.dropna(subset=["unitox_inchikey"])

# If an InChIKey has conflicting external categories,
# mark it ambiguous instead of choosing one arbitrarily.
def combine_categories(values):
    values = sorted(set(values.dropna().astype(str)))
    return values[0] if len(values) == 1 else "Conflicting"

external_unique = (
    external.groupby("unitox_inchikey")["DILIConcern_clean"]
    .agg(combine_categories)
    .reset_index()
)

external_unique["external_reference_label"] = (
    external_unique["DILIConcern_clean"]
    .map({"No": 0, "Most": 1})
)

# Remove any previous external columns before rerunning the merge
cols_to_remove = [
    "DILIConcern_clean",
    "unitox_inchikey",
    "external_reference_label",
    "A_external_agreement"
]
oof_df = oof_df.drop(
    columns=[c for c in cols_to_remove if c in oof_df.columns]
)

rows_before = len(oof_df)

oof_df = oof_df.merge(
    external_unique,
    left_on="inchikey",
    right_on="unitox_inchikey",
    how="left",
    validate="many_to_one"
)

assert len(oof_df) == rows_before, "Merge changed the number of rows!"

# Assign agreement only for clear reference categories
eligible = oof_df["external_reference_label"].notna()

oof_df["A_external_agreement"] = np.nan
oof_df.loc[eligible, "A_external_agreement"] = (
    oof_df.loc[eligible, TARGET].astype(int)
    == oof_df.loc[eligible, "external_reference_label"].astype(int)
).astype(int)

print("Rows retained:", len(oof_df))
print("Rows with clear external reference:", int(eligible.sum()))
print("External agreements:",
      int((oof_df["A_external_agreement"] == 1).sum()))
print("External disagreements:",
      int((oof_df["A_external_agreement"] == 0).sum()))
print("Unassigned external signals:",
      int(oof_df["A_external_agreement"].isna().sum()))

display(
    oof_df.groupby("DILIConcern_clean", dropna=False)[
        "A_external_agreement"
    ].agg(["count", "mean"])
)


Rows retained: 1349
Rows with clear external reference: 116
External agreements: 100
External disagreements: 16
Unassigned external signals: 1233


,count,mean
DILIConcern_clean,,
Less,0,NaN
Most,55,0.963636
No,61,0.770492
<NA>,0,NaN


In [52]:

# Evaluate model confidence against external agreement
external_eval = oof_df[
    oof_df["A_external_agreement"].notna()
].copy()

external_eval["external_agreement_label"] = (
    external_eval["A_external_agreement"].astype(int)
)

print("Eligible drugs:", len(external_eval))

print("\nModel confidence by external agreement:")
display(
    external_eval.groupby("external_agreement_label")[
        "oof_model_confidence"
    ].agg(["count", "mean", "median", "std"])
    .rename(index={
        0: "Disagrees with external reference",
        1: "Agrees with external reference"
    })
)

print("\nModel agreement with UniTox by external agreement:")
display(
    external_eval.groupby("external_agreement_label")[
        "oof_agrees_with_label"
    ].agg(["count", "mean"])
    .rename(index={
        0: "Disagrees with external reference",
        1: "Agrees with external reference"
    })
)

print("\nCross-tabulation:")
display(pd.crosstab(
    external_eval["external_agreement_label"],
    external_eval["oof_agrees_with_label"],
    rownames=["External agreement"],
    colnames=["Model agrees with UniTox"]
))


Eligible drugs: 116

Model confidence by external agreement:


,count,mean,median,std
external_agreement_label,,,,
Disagrees with external reference,16,0.27875,0.296,0.139857
Agrees with external reference,100,0.30494,0.270,0.190654



Model agreement with UniTox by external agreement:


,count,mean
external_agreement_label,,
Disagrees with external reference,16,0.625
Agrees with external reference,100,0.690



Cross-tabulation:


Model agrees with UniTox,False,True
External agreement,,
0,6,10
1,31,69


In [53]:

from scipy.stats import fisher_exact, chi2_contingency
from sklearn.metrics import cohen_kappa_score

# Rows: external agreement (0/1)
# Columns: model agrees with UniTox (False/True)
ct = pd.crosstab(
    external_eval["external_agreement_label"],
    external_eval["oof_agrees_with_label"]
).reindex(index=[0, 1], columns=[False, True], fill_value=0)

print("Contingency table:")
display(ct)

# Fisher exact test for association
odds_ratio, p_value = fisher_exact(ct.to_numpy())

print(f"Odds ratio: {odds_ratio:.4f}")
print(f"Fisher exact p-value: {p_value:.6g}")

# Descriptive agreement between the two binary indicators
kappa = cohen_kappa_score(
    external_eval["external_agreement_label"].astype(int),
    external_eval["oof_agrees_with_label"].astype(int)
)

print(f"Cohen's kappa: {kappa:.4f}")


Contingency table:


oof_agrees_with_label,False,True
external_agreement_label,,
0,6,10
1,31,69


Odds ratio: 1.3355
Fisher exact p-value: 0.578676
Cohen's kappa: 0.0419
